# Embeddings

This lab is designed to help you solidify your understanding of embeddings by applying them to tasks like semantic similarity, clustering, and building a semantic search system.

### Tasks:
- Task 1: Semantic Similarity Comparison
- Task 2: Document Clustering
- Task 3: Enhance the Semantic Search System


## Setup

On my Mac `KMeans` crashed with `OMP: Error #179` as soon as sentence-transformers (torch) was imported, because torch and scikit-learn both bring their own OpenMP library. Setting the number of threads to 1 **before** the imports fixes it. KMeans gets a bit slower, but for these sizes it does not matter.

In [1]:
import os
os.environ["OMP_NUM_THREADS"] = "1"            # torch + scikit-learn OpenMP clash on macOS
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)

## Task 1: Semantic Similarity Comparison
### Objective:
Compare semantic similarity between pairs of sentences using cosine similarity and embeddings.

### Steps:
1. Load a pre-trained Sentence Transformer model.
2. Encode the sentence pairs.
3. Compute cosine similarity for each pair.

### Dataset:
- "A dog is playing in the park." vs. "A dog is running in a field."
- "I love pizza." vs. "I enjoy ice cream."
- "What is AI?" vs. "How does a computer learn?"


In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load pre-trained model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sentence pairs
sentence_pairs = [
    ("A dog is playing in the park.", "A dog is running in a field."),
    ("I love pizza.", "I enjoy ice cream."),
    ("What is AI?", "How does a computer learn?")
]

# Compute similarities
rows = []
for a, b in sentence_pairs:
    emb = model.encode([a, b])
    rows.append({"sentence 1": a, "sentence 2": b,
                 "cosine similarity": cosine_similarity([emb[0]], [emb[1]])[0, 0]})

similarities = pd.DataFrame(rows).sort_values("cosine similarity", ascending=False)
similarities.round(3)

W1109 18:53:55.569000 51700 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Could not fetch README.md from 'sentence-transformers/all-MiniLM-L6-v2' (TypeError: Decompressor.decompress() got an unexpected keyword argument 'output_buffer_limit'). Model will load without a model card.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,sentence 1,sentence 2,cosine similarity
1,I love pizza.,I enjoy ice cream.,0.528
0,A dog is playing in the park.,A dog is running in a field.,0.522
2,What is AI?,How does a computer learn?,0.319


Just to be sure what `cosine_similarity` does, the same number by hand for the first pair: the dot product divided by the product of the two vector lengths. It only looks at the angle between the vectors, not at their length.

In [ ]:
a, b = model.encode(sentence_pairs[0])
print("Embedding length:", a.shape[0])
print("By hand:     ", round(float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b))), 3))
print("scikit-learn:", round(float(cosine_similarity([a], [b])[0, 0]), 3))

Embedding length: 384
By hand:      0.522
scikit-learn: 0.522


Both give the same value. Every sentence becomes a vector of 384 numbers.

### Questions:
- Which sentence pairs are the most semantically similar? Why?
- Can you think of cases where cosine similarity might fail to capture true semantic meaning?


**Which sentence pairs are the most semantically similar? Why?**

The pizza / ice cream pair and the dog pair are basically tied (0.528 and 0.522). The AI pair is clearly behind with 0.319.

I expected the dog pair to win clearly, since both sentences have the same subject (a dog), almost the same action (playing / running) and the same kind of place (park / field). Probably the pizza pair catches up because both sentences have exactly the same structure (*I* + a verb of liking + a food) and *love* and *enjoy* are near-synonyms. So the model sees them as the same kind of statement, just about a different food. The AI pair does not share a single word and the two questions are only connected through the topic (AI ≈ computers that learn). The model picks up some of that, but not much.

Also none of the values is close to 1. Two different sentences with `all-MiniLM-L6-v2` rarely score above ~0.6, so the absolute number says less than the ranking between the pairs.

**Can you think of cases where cosine similarity might fail to capture true semantic meaning?**

Testing a few cases where I expect it to fail:

In [ ]:
tricky_pairs = [
    ("The service was bad.", "The service was not bad."),                # negation
    ("Dog bites man.", "Man bites dog."),                                # word order
    ("The movie was great.", "The movie was terrible."),                 # opposite opinion
    ("I love pizza.", "I hate pizza."),                                  # opposite opinion
    ("I deposited cash at the bank.", "We had a picnic on the river bank."),  # same word, other meaning
    ("I deposited cash at the bank.", "The ATM swallowed my card."),     # no shared word, same topic
]

rows = []
for a, b in tricky_pairs:
    emb = model.encode([a, b])
    rows.append({"sentence 1": a, "sentence 2": b, "cosine similarity": cosine_similarity(emb)[0, 1]})
pd.DataFrame(rows).round(3)

,sentence 1,sentence 2,cosine similarity
0,The service was bad.,The service was not bad.,0.907
1,Dog bites man.,Man bites dog.,0.881
2,The movie was great.,The movie was terrible.,0.666
3,I love pizza.,I hate pizza.,0.798
4,I deposited cash at the bank.,We had a picnic on the river bank.,0.386
5,I deposited cash at the bank.,The ATM swallowed my card.,0.531


Apparently the model mostly measures whether two sentences are **about the same thing**, not whether they **say the same thing**:

- **Negation:** *bad* and *not bad* score ~0.91, although they are opposite opinions. The one word *not* hardly moves the vector.
- **Word order:** *dog bites man* and *man bites dog* score ~0.88. A sentence model does see the order (averaged word vectors like GloVe would give exactly 1.0), but only a little.
- **Opposite opinions:** *love pizza* / *hate pizza* still scores ~0.80, higher than any of the three lab pairs. Same topic, opposite meaning.
- **Ambiguous words:** here the model actually does it right. The two *bank* sentences share a word but score lower (~0.39) than the ATM sentence that shares no word at all (~0.53).

Other cases where it can fail: sarcasm, numbers and facts (*the meeting is at 3pm* vs *the meeting is at 5pm* are almost identical vectors), texts longer than the model's limit of 256 tokens (the rest is just cut off), words that did not exist in the training data and domain language the model never saw (legal, medical abbreviations). And the embeddings carry the biases of the text they were trained on.

So a high cosine similarity means *related*, not *equal*. For search that is usually fine, for sentiment or checking whether two statements agree it is not.

## Task 2: Document Clustering
### Objective:
Cluster a set of text documents into similar groups based on their embeddings.

### Steps:
1. Encode the documents using Sentence Transformers.
2. Use KMeans clustering to group the documents.
3. Analyze the clusters for semantic meaning.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# Documents to cluster
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?"
]

# Encode documents
# normalize_embeddings=True scales every vector to length 1. Then the euclidean distance
# that KMeans uses is directly linked to the cosine similarity: |a - b|² = 2 - 2·cos(a, b)
embeddings = model.encode(documents, normalize_embeddings=True)
embeddings.shape

(6, 384)

Before clustering, how similar are the six documents to each other at all?

In [ ]:
pd.DataFrame(cosine_similarity(embeddings), index=documents, columns=range(len(documents))).round(2)

,0,1,2,3,4,5
What is the capital of France?,1.00,0.13,0.19,0.07,0.12,0.12
How do I bake a chocolate cake?,0.13,1.00,0.16,0.18,0.06,0.07
What is the distance between Earth and Mars?,0.19,0.16,1.00,0.15,0.08,0.04
How do I change a flat tire on a car?,0.07,0.18,0.15,1.00,0.09,0.19
What is the best way to learn Python?,0.12,0.06,0.08,0.09,1.00,0.09
How do I fix a leaky faucet?,0.12,0.07,0.04,0.19,0.09,1.00


All pairs are between 0.04 and 0.19. So there are no two documents that are clearly closer to each other than to the rest. Every question is about a different topic. That already makes me doubt there are real clusters here.

In [ ]:
# Perform KMeans clustering
# With 6 documents k can only go from 2 to 5 for the silhouette score
scores = []
for k in range(2, len(documents)):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(embeddings)
    scores.append({"k": k,
                   "silhouette": silhouette_score(embeddings, km.labels_, metric="cosine"),
                   "inertia": km.inertia_})
pd.DataFrame(scores).set_index("k").round(3)

,silhouette,inertia
k,,
2,0.048,3.435
3,0.057,2.488
4,0.041,1.615
5,0.019,0.807


The silhouette score is highest for k = 3 (0.057), but every value is close to 0. A silhouette close to 0 means a document is about as close to the other clusters as to its own, so none of these clusterings is really convincing. The inertia just drops steadily with no clear elbow. Going with k = 3 since it is the best of the options:

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=0).fit(embeddings)

In [ ]:
# Print cluster assignments
for cluster in sorted(set(kmeans.labels_)):
    print(f"Cluster {cluster}:")
    for doc in np.array(documents)[kmeans.labels_ == cluster]:
        print("   ", doc)

Cluster 0:
    How do I change a flat tire on a car?
    How do I fix a leaky faucet?
Cluster 1:
    What is the capital of France?
    How do I bake a chocolate cake?
    What is the distance between Earth and Mars?
Cluster 2:
    What is the best way to learn Python?


### Questions:
- How many clusters make the most sense? Why?
- Examine the documents in each cluster. Are they semantically meaningful? Can you assign a semantic "theme" to each cluster?
- Try this exercise with a larger dataset of your choice

**How many clusters make the most sense? Why?**

By the numbers k = 3 (highest silhouette, 0.057). Honestly though, with these six documents no k really makes sense. Every document is about its own topic (geography, baking, space, cars, programming, plumbing) and all pairwise similarities are below 0.2. If I had to group them by hand I would either say 6 clusters (one per document, which is no clustering at all) or 2 clusters by the form of the question: *What is ...?* (facts) vs *How do I ...?* (instructions). KMeans does not find that second split though. With k = 2 it puts France, cake and Mars together and tire, Python and faucet together, which is neither.

**Are the clusters semantically meaningful? Can you assign a theme?**

Only partly:

- **tire + faucet:** a real theme, *fixing / repairing things*. Both are practical how-to questions about something that is broken.
- **Python:** alone in its cluster. *Learning / programming*, nothing else in the list is close to it.
- **France + cake + Mars:** no common theme I can see. This is the cluster of what is left over.

So 2 of 3 clusters can be named, but that is because one of them has a single document in it. Six documents are simply too few to say anything about clusters. That's why I tried it with a larger dataset below.

### Larger dataset: 20 Newsgroups

I used the **20 Newsgroups** dataset from scikit-learn (old forum posts from the 90s, each post belongs to one forum). I picked 6 forums that are about clearly different things: computer graphics, cars, ice hockey, medicine, space and Middle East politics. Headers, footers and quotes are removed, so the forum name cannot leak through the email header. Posts with less than 20 characters are dropped since there is nothing to embed.

The forum is **not** used for the clustering. It only comes back at the end to check whether the clusters match the forums.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

forums = ["comp.graphics", "rec.autos", "rec.sport.hockey", "sci.med", "sci.space", "talk.politics.mideast"]
news = fetch_20newsgroups(subset="all", categories=forums, remove=("headers", "footers", "quotes"), random_state=0)

posts = pd.DataFrame({"text": news.data, "forum": [news.target_names[t].split(".")[-1] for t in news.target]})
posts = posts[posts["text"].str.strip().str.len() > 20].reset_index(drop=True)
print(posts.shape)
posts["forum"].value_counts()

(5663, 2)


forum
hockey      969
med         955
space       952
graphics    950
autos       926
mideast     911
Name: count, dtype: int64

5,663 posts, roughly 950 per forum. Encoding them (takes ~10 seconds on a laptop CPU):

In [ ]:
post_embeddings = model.encode(posts["text"].tolist(), normalize_embeddings=True, batch_size=64)
post_embeddings.shape

(5663, 384)

Which k? Same as before, silhouette and inertia (elbow) for k = 2 to 10. Since I know the real forums here, I also print the **adjusted Rand index (ARI)**: 1 means the clusters match the forums exactly, ~0 means random. In a real project you would not have this column.

In [ ]:
scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(post_embeddings)
    scores.append({"k": k,
                   "silhouette": silhouette_score(post_embeddings, km.labels_, metric="cosine"),
                   "inertia": km.inertia_,
                   "ARI vs. forums": adjusted_rand_score(posts["forum"], km.labels_)})
pd.DataFrame(scores).set_index("k").round(3)

,silhouette,inertia,ARI vs. forums
k,,,
2,0.067,5195.222,0.117
3,0.085,5009.855,0.254
4,0.095,4891.715,0.403
5,0.110,4791.844,0.587
6,0.118,4713.767,0.690
7,0.119,4663.987,0.688
8,0.106,4616.639,0.659
9,0.081,4589.846,0.597
10,0.082,4562.254,0.571


The silhouette rises up to k = 6 and 7 (0.118 and 0.119, basically the same) and drops after that. So without knowing anything about the forums the silhouette would have pointed to 6 or 7 clusters, which is about right. The inertia again has no clear elbow, it just gets flatter. The ARI is also highest at k = 6 (0.69).

The silhouette values are still low (~0.12). Real text does not form tight, separated groups: a post about car stereo software is a bit graphics and a bit cars. Comparing the clusters with the forums for k = 6:

In [ ]:
kmeans_posts = KMeans(n_clusters=6, n_init=10, random_state=0).fit(post_embeddings)
posts["cluster"] = kmeans_posts.labels_
pd.crosstab(posts["cluster"], posts["forum"])

forum,autos,graphics,hockey,med,mideast,space
cluster,,,,,,
0,12,0,4,2,783,4
1,788,8,2,2,0,28
2,14,8,5,806,8,19
3,3,788,2,10,1,25
4,0,0,867,0,3,1
5,109,146,89,135,116,875


Five clusters are almost pure: Middle East (783 of 911 posts), autos (788), medicine (806), graphics (788) and hockey (867). The sixth cluster is space (875) plus everything the other clusters didn't take: between 89 and 146 posts from every other forum.

To see what is inside each cluster without looking at the forum, the three posts closest to each cluster centre:

In [ ]:
def snippet(text, length=90):
    return " ".join(text.split())[:length]

for cluster in range(kmeans_posts.n_clusters):
    members = posts[posts["cluster"] == cluster]
    closest = np.argsort(post_embeddings[members.index] @ kmeans_posts.cluster_centers_[cluster])[::-1][:3]
    print(f"Cluster {cluster} ({len(members)} posts)")
    for text in members["text"].iloc[closest]:
        print("   ", snippet(text))

Cluster 0 (805 posts)
    Hate to be simple minded about this Tim, but I think its really very simple. He was a dirt
    I really disagree with you. But maybe you know better. Here is the issue at hand: 'After a
    ^^^^^^^^^^^^^^^^^^ O.K., its my turn: DRIVING THE JEWS INTO THE SEA ?! I am sick and tired
Cluster 1 (828 posts)
    While I don't read normally read this group, I was looking for Valentine radar information
    Drivel. I received delivery of my '93 Trans Am 7 weeks after I ordered (promised 6-8 weeks
    Especially the '68 Shelby-American GT-500KR (King of the Road, so named to steal GM's plan
Cluster 2 (860 posts)
    No doubt the sci.med* folks are getting out their flamethrowers. I'm rather certain that t
    Hate to wreck your elaborate theory, but Steve Dyer is not an MD. So professional jealosy 
    Many people responded with more anecdotal stories; I think its safe to say the original po
Cluster 3 (829 posts)
    Corel Draw 4 will be able to do this as it will inclu

The themes are easy to name from the posts alone: 0 Israel / Arab politics, 1 cars (Trans Am, Shelby GT-500), 2 medicine discussions, 3 image software (Corel Draw, TIFF, 24-bit images), 4 hockey standings and awards. Cluster 5 has space articles but also quoted mail addresses and newspaper references. Investigating that leftover cluster a bit more with k = 7:

In [ ]:
km7 = KMeans(n_clusters=7, n_init=10, random_state=0).fit(post_embeddings)
words = posts["text"].str.split().str.len()
table = pd.crosstab(km7.labels_, posts["forum"])
table["median words"] = words.groupby(km7.labels_).median()
table

forum,autos,graphics,hockey,med,mideast,space,median words
row_0,,,,,,,
0,1,0,0,1,726,0,183.5
1,3,785,2,9,1,26,65.5
2,6,5,3,752,3,12,105.0
3,121,128,107,178,176,199,45.0
4,0,0,854,0,2,0,99.0
5,17,22,1,11,3,688,102.0
6,778,10,2,4,0,27,83.0


With k = 7 space gets its own cleaner cluster (688 posts) and one cluster (3) becomes a mix of every forum with ~110 to ~200 posts each. Its posts are also the shortest (median 45 words, the other clusters have 66 to 184). Probably those are short replies like *"thanks, that helped"* or *"I agree"* that don't really belong to any topic, so the model puts them close to each other. That is a group that exists in the data but not in the forum labels, which is kind of the point of clustering.

Last check: the same KMeans on TF-IDF vectors (counting words instead of meaning):

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

post_tfidf_small = TfidfVectorizer(stop_words="english", min_df=2).fit_transform(posts["text"])
labels_tfidf = KMeans(n_clusters=6, n_init=10, random_state=0).fit_predict(post_tfidf_small)

print(f"ARI, embeddings: {adjusted_rand_score(posts['forum'], kmeans_posts.labels_):.2f}")
print(f"ARI, TF-IDF:     {adjusted_rand_score(posts['forum'], labels_tfidf):.2f}")

ARI, embeddings: 0.69
ARI, TF-IDF:     0.20


The embedding clusters match the forums a lot better (0.69 vs 0.20). Most posts share only a few words with each other, so in TF-IDF space the distances between them are all similar and KMeans can't separate much.

**Summary for the larger dataset:** 6 (or 7) clusters make the most sense, the silhouette finds that without knowing the forums, and 5 of 6 clusters have a clear theme. The 6th is a mix of space posts and short off-topic replies. On the six lab documents this did not work because there is no group structure to find in six unrelated questions.

## Task 3: Semantic Search System
### Objective:
Create a semantic search engine:
A user provides a query and you search the dataset for semantically relevant documents to return. Return the top 5 results.

### Dataset:
- Use the following set of documents:
    - "What is the capital of France?"
    - "How do I bake a chocolate cake?"
    - "What is the distance between Earth and Mars?"
    - "How do I change a flat tire on a car?"
    - "What is the best way to learn Python?"
    - "How do I fix a leaky faucet?"
    - "What are the best travel destinations in Europe?"
    - "How do I set up a local server?"
    - "What is quantum computing?"
    - "How do I build a mobile app?"


In [ ]:
# Documents dataset
documents = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is the distance between Earth and Mars?",
    "How do I change a flat tire on a car?",
    "What is the best way to learn Python?",
    "How do I fix a leaky faucet?",
    "What are the best travel destinations in Europe?",
    "How do I set up a local server?",
    "What is quantum computing?",
    "How do I build a mobile app?"
]

# Compute document embeddings
# computed once, every query is then compared against these vectors
doc_embeddings = model.encode(documents, normalize_embeddings=True)
doc_embeddings.shape

(10, 384)

In [ ]:
# Create the search function
#This function should encode the user query and return the top N documents that most resemble it
def semantic_search(query, documents, doc_embeddings, top_n=5):
    # all vectors have length 1, so the dot product is the cosine similarity
    query_embedding = model.encode(query, normalize_embeddings=True)
    scores = doc_embeddings @ query_embedding
    top = np.argsort(scores)[::-1][:top_n]          # positions of the top_n highest scores
    return pd.DataFrame({
        "score": scores[top].round(3),
        "document": [documents[i] for i in top],
    }, index=pd.RangeIndex(1, len(top) + 1, name="rank"))

In [ ]:
# Test the search function
query = "Explain programming languages."
semantic_search(query, documents, doc_embeddings)

,score,document
rank,,
1,0.435,What is quantum computing?
2,0.319,What is the best way to learn Python?
3,0.110,How do I build a mobile app?
4,0.091,How do I set up a local server?
5,0.091,What are the best travel destinations in Europe?


A few more queries, written the way a person would ask:

In [ ]:
for query in ["How do I repair things at home?", "Where should I go on holiday?", "dessert recipes", "Tell me about space"]:
    print(f"Query: {query!r}")
    display(semantic_search(query, documents, doc_embeddings))

Query: 'How do I repair things at home?'


,score,document
rank,,
1,0.320,How do I fix a leaky faucet?
2,0.214,How do I change a flat tire on a car?
3,0.179,How do I set up a local server?
4,0.168,What is the best way to learn Python?
5,0.145,How do I build a mobile app?


Query: 'Where should I go on holiday?'


,score,document
rank,,
1,0.476,What are the best travel destinations in Europe?
2,0.192,What is the capital of France?
3,0.161,What is the best way to learn Python?
4,0.132,How do I bake a chocolate cake?
5,0.092,What is the distance between Earth and Mars?


Query: 'dessert recipes'


,score,document
rank,,
1,0.495,How do I bake a chocolate cake?
2,0.123,How do I build a mobile app?
3,0.120,What are the best travel destinations in Europe?
4,0.105,What is the distance between Earth and Mars?
5,0.076,How do I change a flat tire on a car?


Query: 'Tell me about space'


,score,document
rank,,
1,0.341,What is the distance between Earth and Mars?
2,0.232,What is quantum computing?
3,0.138,What is the capital of France?
4,0.125,What is the best way to learn Python?
5,0.121,What are the best travel destinations in Europe?


### Questions:
- What are the top-ranked results for the given queries?
- How can you improve the ranking explanation for users?
- Try this approach with a larger dataset

**What are the top-ranked results for the given queries?**

For *"Explain programming languages."*:

1. *What is quantum computing?* (0.435)
2. *What is the best way to learn Python?* (0.319)
3. *How do I build a mobile app?* (0.110)
4. *How do I set up a local server?* (0.091)
5. *What are the best travel destinations in Europe?* (0.091)

I expected Python on rank 1. Probably quantum computing wins because *"Explain X"* and *"What is X?"* are both "explain a concept to me" questions and *computing* is close to *programming*, while the Python question is about how to **learn** something. After rank 2 the score drops to ~0.1, and ranks 3 to 5 are basically noise. *Travel destinations in Europe* in the top 5 shows the problem: `top_n=5` always returns 5 documents, even if only 2 are relevant.

The other queries work well: *repair things at home* → faucet (0.32) and flat tire (0.21), both without a shared word. *Holiday* → travel destinations (0.48). *Dessert recipes* → chocolate cake (0.50). *Space* → Earth and Mars (0.34). Every time the right document is on rank 1, and there's a clear gap to rank 2 or 3.

**How can you improve the ranking explanation for users?**

A plain list of 5 documents hides two things from the user: how sure the system is and why a document was found. What I would add:

- **show the score as a label** (strong / related / weak) instead of a raw number like 0.091 that means nothing to a user
- **hide results below a minimum score**, and say how many were hidden, instead of always showing 5
- **show the shared words**, so the user sees whether a result was found by the same words or only by meaning (*repair* → *fix a leaky faucet*)
- with longer documents, **show the best-matching sentence or chunk** instead of the whole document

The thresholds below (0.4 and 0.2) are picked by hand from the results above. With only 10 short documents that is a guess: with longer documents the scores are generally lower, so they would have to be set again on real queries.

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

def match_label(score):
    if score >= 0.4:
        return "strong match"
    if score >= 0.2:
        return "related"
    return "weak"

def shared_words(query, document):
    tokens = lambda text: set(re.findall(r"[a-z]+", text.lower())) - ENGLISH_STOP_WORDS
    shared = tokens(query) & tokens(document)
    return ", ".join(sorted(shared)) if shared else "none, matched by meaning"

def explained_search(query, documents, doc_embeddings, top_n=5, min_score=0.2):
    results = semantic_search(query, documents, doc_embeddings, top_n)
    results["match"] = results["score"].map(match_label)
    results["shared words"] = results["document"].map(lambda doc: shared_words(query, doc))
    shown = results[results["score"] >= min_score]
    print(f"Query: {query!r}  ({len(results) - len(shown)} weaker results hidden)")
    return shown

for query in ["Explain programming languages.", "How do I repair things at home?", "Tell me about space"]:
    display(explained_search(query, documents, doc_embeddings))

Query: 'Explain programming languages.'  (3 weaker results hidden)


,score,document,match,shared words
rank,,,,
1,0.435,What is quantum computing?,strong match,"none, matched by meaning"
2,0.319,What is the best way to learn Python?,related,"none, matched by meaning"


Query: 'How do I repair things at home?'  (3 weaker results hidden)


,score,document,match,shared words
rank,,,,
1,0.320,How do I fix a leaky faucet?,related,"none, matched by meaning"
2,0.214,How do I change a flat tire on a car?,related,"none, matched by meaning"


Query: 'Tell me about space'  (3 weaker results hidden)


,score,document,match,shared words
rank,,,,
1,0.341,What is the distance between Earth and Mars?,related,"none, matched by meaning"
2,0.232,What is quantum computing?,related,"none, matched by meaning"


Now *"Explain programming languages."* shows 2 results instead of 5 and the user is told that 3 weak ones were hidden. *"Tell me about space"* still shows quantum computing as *related* (0.23), which a person would not call related. So a fixed threshold is not perfect. Better ranking itself (not only better explanation) would come from:

- **re-ranking** the top candidates with a stronger (cross-encoder) model that reads query and document together
- **hybrid search:** combining the embedding score with a keyword score (TF-IDF / BM25), which helps when the user types exact names or codes
- **a better embedding model** (e.g. `all-mpnet-base-v2`) if the quality is not good enough, at the cost of speed

**Try this approach with a larger dataset**

Reusing the 5,663 forum posts and their embeddings from Task 2. Since `post_embeddings` are already computed (that's the "index"), a search is only one matrix product. To compare, the same search with TF-IDF vectors: `TfidfVectorizer` also normalises every row to length 1, so the dot product is again the cosine similarity.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
post_tfidf = tfidf.fit_transform(posts["text"])

def search_posts(query, top_n=5, method="embeddings"):
    if method == "embeddings":
        scores = post_embeddings @ model.encode(query, normalize_embeddings=True)
    else:
        scores = (post_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    top = np.argsort(scores)[::-1][:top_n]
    return pd.DataFrame({
        "score": scores[top].round(2),
        "forum": posts["forum"].iloc[top].values,
        "post": [snippet(t, 80) for t in posts["text"].iloc[top]],
    }, index=pd.RangeIndex(1, len(top) + 1, name="rank"))

for query in ["best software to edit photos", "goalkeeper made a great save"]:
    print(f"Query: {query!r}\nEmbeddings")
    display(search_posts(query))
    print("TF-IDF")
    display(search_posts(query, method="tfidf"))

Query: 'best software to edit photos'
Embeddings


,score,forum,post
rank,,,
1,0.54,graphics,"Check out Image Pals v1.2 from U-Lead (until May, special $99 intro price, 310-5"
2,0.53,graphics,"Hi there, I am looking for advice on software/hardware package for making, stori"
3,0.52,graphics,I am looking for comments from people who have used/heard about PhotoShop for Wi
4,0.47,graphics,"Hello, I have been searching for a quality image enhancement and manipulation pa"
5,0.46,graphics,An review of both the Mac and Windows versions in either PC Week or Info World t


TF-IDF


,score,forum,post
rank,,,
1,0.26,space,Nick Haines sez; >(given that I've heard the Shuttle software rated as Level 5 i
2,0.24,graphics,"Hi there, I am looking for advice on software/hardware package for making, stori"
3,0.19,graphics,"Hi there, I am here looking for some help. My friend is a interior decor designe"
4,0.17,mideast,"Yes, it is. I have taken photos of it's minaret. Dunno. This doesn't sound like"
5,0.17,space,"According to a Software engineering professor here, what was actually rated leve"


Query: 'goalkeeper made a great save'
Embeddings


,score,forum,post
rank,,,
1,0.50,hockey,Whoa. What you are effectively doing is implying that if a player plays really w
2,0.47,hockey,"(Detroit, April 21) Most knowledgable observers once again watched in shock as t"
3,0.45,hockey,Aargh! Paul Stewart is the worst and most biased ref. presently in the NHL. He c
4,0.44,hockey,The overhead replay pretty clearly showed that it was the Quebec defender who de
5,0.43,hockey,The goalie to whom you refer is Clint Malarchuk. He was at that time playing wit


TF-IDF


,score,forum,post
rank,,,
1,0.13,hockey,"While I will first congratulate the Blues for their victory, I think it must be"
2,0.11,graphics,"Sorry for the repeat of this request, but does anyone know of a good free/sharew"
3,0.10,hockey,YAHOOOOOOOOO! The Habs tied this sucker at 2 and the teams now head for Quebec C
4,0.10,hockey,This game would have been great as part of a double-header on ABC or ESPN; the l
5,0.10,hockey,"I hope this makes you feel better, but the overtime between the Isles and the Ca"


For *best software to edit photos* the embedding search returns only graphics posts about image software. TF-IDF puts a **space post** on rank 1: it is about the *software* of the space shuttle, and *software* is the only query word it found. It doesn't know that *photos* has something to do with *image editing*. For *goalkeeper made a great save* the embedding search finds only hockey posts, although hockey fans write *goalie* and not *goalkeeper*. TF-IDF has a graphics post on rank 2 that asks how to *save* ray-traced images as files: the right word with the wrong meaning.

Two examples are not a test though. Checking it on 12 queries where I know which forum the answer should come from, and counting how many of the top 5 results are from that forum (precision at 5):

In [ ]:
test_queries = {
    "my engine makes a strange noise when I brake": "autos",
    "which car should I buy as a student": "autos",
    "symptoms of a migraine": "med",
    "is it dangerous to take antibiotics too often": "med",
    "astronauts living on a station in orbit": "space",
    "launching a satellite to Mars": "space",
    "best software to edit photos": "graphics",
    "how to render a 3D scene": "graphics",
    "goalkeeper made a great save": "hockey",
    "who will win the cup this year": "hockey",
    "peace talks between countries": "mideast",
    "the conflict over land and borders": "mideast",
}

rows = []
for query, forum in test_queries.items():
    rows.append({"query": query, "expected forum": forum,
                 "embeddings": (search_posts(query)["forum"] == forum).mean(),
                 "TF-IDF": (search_posts(query, method="tfidf")["forum"] == forum).mean()})
evaluation = pd.DataFrame(rows)
display(evaluation)
evaluation[["embeddings", "TF-IDF"]].mean().round(3)

,query,expected forum,embeddings,TF-IDF
0,my engine makes a strange noise when I brake,autos,1.0,1.0
1,which car should I buy as a student,autos,1.0,1.0
2,symptoms of a migraine,med,1.0,1.0
3,is it dangerous to take antibiotics too often,med,1.0,1.0
4,astronauts living on a station in orbit,space,1.0,0.8
5,launching a satellite to Mars,space,1.0,1.0
6,best software to edit photos,graphics,1.0,0.4
7,how to render a 3D scene,graphics,1.0,1.0
8,goalkeeper made a great save,hockey,1.0,0.8
9,who will win the cup this year,hockey,1.0,1.0


embeddings    1.000
TF-IDF        0.917
dtype: float64

The embedding search gets all 60 results from the right forum (precision 1.0), TF-IDF gets 55 of 60 (0.917). TF-IDF only loses on the queries where the user's words are not the words of the forum (*photos*, *goalkeeper*, *astronauts*). When the query uses the same words as the posts (*migraine*, *Mars*, *peace talks*) TF-IDF is just as good, and it's faster and needs no model. That's why real systems often combine both (hybrid search).

Limits of this check:

- "Right forum" is a rough measure. A hockey post can be from the right forum and still not answer the question. A real evaluation would need someone to judge each result.
- 12 queries that I wrote myself, knowing the forums. Real users ask differently.
- `all-MiniLM-L6-v2` only reads the first 256 tokens (~200 words) of every post and ignores the rest. For long documents they would have to be split into **chunks** and every chunk embedded separately, which is the first step of a RAG system.
- With millions of documents comparing the query with every vector gets too slow. A vector database (Chroma, FAISS) finds the nearest vectors approximately instead.